# Python Mastery — Module 11
# Applied LLM Engineering and Retrieval-Augmented Generation (RAG)

This module bridges NLP fundamentals to practical LLM applications. It covers prompting, structured outputs, embeddings, document chunking, retrieval, reranking, grounded generation, evaluation, guardrails, privacy, cost/latency, and a working local RAG baseline.

## Learning outcomes
- Design prompts with clear instructions, context, constraints, and output formats.
- Use structured outputs and validate model responses.
- Explain hallucinations and grounding.
- Build a retrieval pipeline: ingest → chunk → embed → retrieve → assemble context → answer.
- Establish a lexical baseline and an embedding-based retrieval design.
- Evaluate retrieval and generated answers separately.
- Add citations, abstention, prompt-injection defenses, and privacy safeguards.
- Think about latency, token cost, observability, and production readiness.

**Important:** The notebook is provider-neutral. The runnable core uses local TF-IDF retrieval and a deterministic grounded-answer baseline so it works without API keys. Optional hosted-LLM calls are shown as a pattern, not executed.

## 1. What is an LLM application?

A language model generates text based on its input and learned parameters. An LLM application wraps that model with the surrounding system needed to solve a task reliably:
- user interface and input validation;
- prompt templates and conversation state;
- retrieval or tools when external/current information is needed;
- output parsing and validation;
- authorization, privacy, safety, and policy checks;
- evaluation, logging, monitoring, and fallback behavior.

A strong model alone does not guarantee a reliable product. The surrounding system often determines whether answers are grounded, secure, timely, and useful.

## 2. Prompt engineering: specify the job clearly

A practical prompt often defines:
1. **Role/task:** what the model should do.
2. **Context:** facts or documents it may use.
3. **Constraints:** what it must and must not do.
4. **Output schema:** exact structure expected.
5. **Uncertainty behavior:** what to do when information is missing.
6. **Examples:** representative inputs and desired outputs, when useful.

Avoid vague instructions such as “be smart.” Prefer testable constraints. Prompt engineering improves task framing, but it cannot guarantee factual correctness.

In [ ]:
def build_prompt(question, context, output_format="bullet points"):
    return f"""Task: Answer the user's question using only the supplied context.
If the context does not contain the answer, say that the available documents do not provide enough information.
Do not invent facts or citations.

Context:
{context}

Question:
{question}

Output format:
{output_format}
"""

sample_context = "Refund requests must be submitted within 30 calendar days of purchase."
print(build_prompt("How long do I have to request a refund?", sample_context))

## 3. Structured outputs and validation

Applications often need machine-readable output such as JSON. Ask the model for a schema, then validate the result in code. A prompt that says “return valid JSON” is not a substitute for parsing and validation.

Practical rules:
- validate required fields and types;
- reject unexpected values;
- handle malformed output with a controlled retry or fallback;
- never execute arbitrary model-generated code;
- validate any URLs, SQL, tool arguments, or actions before using them;
- keep the model's output separate from trusted application instructions.

In [ ]:
import json
from typing import Any

required_keys = {"answer", "evidence_ids", "insufficient_evidence"}

def validate_answer_payload(payload: Any) -> dict:
    if not isinstance(payload, dict):
        raise ValueError("Payload must be a JSON object.")
    missing = required_keys - set(payload)
    extra = set(payload) - required_keys
    if missing:
        raise ValueError(f"Missing keys: {sorted(missing)}")
    if extra:
        raise ValueError(f"Unexpected keys: {sorted(extra)}")
    if not isinstance(payload["answer"], str):
        raise ValueError("'answer' must be a string.")
    if not isinstance(payload["evidence_ids"], list) or not all(
        isinstance(item, str) for item in payload["evidence_ids"]
    ):
        raise ValueError("'evidence_ids' must be a list of strings.")
    if not isinstance(payload["insufficient_evidence"], bool):
        raise ValueError("'insufficient_evidence' must be boolean.")
    return payload

valid_payload = {
    "answer": "Submit the request within 30 calendar days.",
    "evidence_ids": ["policy_001"],
    "insufficient_evidence": False,
}
validate_answer_payload(valid_payload)

## 4. Hallucination, grounding, and abstention

A hallucination is an answer that presents unsupported or false information as if it were true. It may happen even when the answer sounds fluent and confident.

**Grounding** means connecting claims to an authorized source, such as retrieved documents or a verified database.

Useful design patterns:
- include relevant source excerpts in the prompt;
- ask for evidence IDs or citations for claims;
- check that cited IDs actually exist in retrieved context;
- instruct the system to abstain when evidence is insufficient;
- separate source facts from model-generated interpretation;
- evaluate unsupported claims, not just fluency.

Citations can be fabricated by a model unless the application validates them.

## 5. RAG architecture

Retrieval-Augmented Generation (RAG) retrieves relevant external information and supplies it to an LLM at answer time.

**Indexing time**
1. collect authorized documents;
2. parse and clean them;
3. split them into chunks;
4. attach metadata and source IDs;
5. embed chunks and store the vectors or build a lexical index.

**Query time**
1. validate the user query;
2. retrieve candidate chunks;
3. optionally rerank candidates;
4. build a context with source identifiers;
5. generate a grounded answer;
6. validate evidence and format;
7. return answer plus citations or abstention.

RAG can improve access to private or changing information, but does not guarantee correctness. Retrieval can miss the answer, documents can be wrong, and the generator can misuse the context.

## 6. Document chunking

Chunking breaks long documents into smaller passages. Chunks should be large enough to preserve meaning but small enough to retrieve precisely.

Common strategies:
- fixed token/character windows with overlap;
- split on paragraphs or headings;
- recursive splitting using separators;
- structure-aware splitting for HTML, Markdown, tables, code, or PDFs.

Overlap can preserve context across boundaries but increases index size and duplicate retrieval. Chunk size and overlap should be tuned using realistic questions and retrieval evaluation, not chosen by habit alone.

In [ ]:
import numpy as np
import pandas as pd

documents = [
    {
        "id": "policy_refund",
        "title": "Refund Policy",
        "text": "Customers may request a refund within 30 calendar days of purchase. Requests must include the order number. Refunds are reviewed within five business days."
    },
    {
        "id": "policy_claim",
        "title": "Claim Submission",
        "text": "A claim submission should include the member identifier, date of service, provider details, and itemized charges. Missing information may delay processing."
    },
    {
        "id": "policy_security",
        "title": "Account Security",
        "text": "Never share your password or one-time verification code. Report suspected account compromise to the security team immediately."
    },
    {
        "id": "policy_payment",
        "title": "Payment Troubleshooting",
        "text": "If a payment fails, verify the billing details and payment method. Do not retry repeatedly if the account shows a pending transaction; contact support for assistance."
    },
]

def chunk_text(text, chunk_size=110, overlap=25):
    if chunk_size <= 0 or overlap < 0 or overlap >= chunk_size:
        raise ValueError("Require chunk_size > overlap >= 0.")
    chunks = []
    start = 0
    while start < len(text):
        end = min(len(text), start + chunk_size)
        chunks.append(text[start:end])
        if end == len(text):
            break
        start = end - overlap
    return chunks

chunk_rows = []
for doc in documents:
    for i, chunk in enumerate(chunk_text(doc["text"], chunk_size=110, overlap=25)):
        chunk_rows.append({
            "chunk_id": f"{doc['id']}_chunk_{i}",
            "document_id": doc["id"],
            "title": doc["title"],
            "text": chunk,
        })
chunks_df = pd.DataFrame(chunk_rows)
chunks_df[["chunk_id", "title", "text"]]

## 7. A runnable retrieval baseline with TF-IDF

Before adding a vector database, create a simple lexical retrieval baseline. TF-IDF compares query terms with document chunks and is fast, interpretable, and easy to run locally.

Limitations:
- exact or related vocabulary helps; paraphrases may be missed;
- lexical overlap does not guarantee relevance;
- the score is not a calibrated probability of relevance.

An embedding-based retriever can improve semantic matching, but should be evaluated against a baseline. Hybrid retrieval combines lexical and semantic candidates.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vectorizer = TfidfVectorizer(ngram_range=(1, 2), stop_words="english")
chunk_matrix = vectorizer.fit_transform(chunks_df["text"])

def retrieve(query, top_k=3):
    query_vector = vectorizer.transform([query])
    scores = cosine_similarity(query_vector, chunk_matrix).ravel()
    order = np.argsort(scores)[::-1][:top_k]
    return [
        {
            "chunk_id": chunks_df.iloc[i]["chunk_id"],
            "document_id": chunks_df.iloc[i]["document_id"],
            "title": chunks_df.iloc[i]["title"],
            "text": chunks_df.iloc[i]["text"],
            "score": float(scores[i]),
        }
        for i in order
    ]

retrieve("How many days do I have to request a refund?", top_k=3)

## 8. Inspect retrieval quality before generating an answer

If retrieval does not find the right evidence, the generator cannot reliably cite it. Test retrieval with a small set of representative queries and known relevant document IDs.

Common retrieval metrics:
- **Recall@k:** whether a relevant item appears in the top k.
- **Precision@k:** proportion of top-k items that are relevant.
- **MRR:** rewards placing the first relevant result high.
- **nDCG:** handles ranked and potentially graded relevance.

These metrics measure retrieval, not the factuality of the final generated answer.

In [ ]:
# Tiny labeled evaluation set. The relevance labels are hand-authored for this toy corpus.
eval_queries = [
    {"query": "refund deadline after purchase", "relevant_doc_ids": {"policy_refund"}},
    {"query": "what details are required for a claim", "relevant_doc_ids": {"policy_claim"}},
    {"query": "what if payment is pending", "relevant_doc_ids": {"policy_payment"}},
    {"query": "how to protect my password", "relevant_doc_ids": {"policy_security"}},
]

def recall_at_k(query, relevant_ids, k=3):
    retrieved_ids = {item["document_id"] for item in retrieve(query, top_k=k)}
    return len(retrieved_ids & relevant_ids) / len(relevant_ids) if relevant_ids else np.nan

eval_results = []
for item in eval_queries:
    eval_results.append({
        "query": item["query"],
        "relevant_doc_ids": ", ".join(sorted(item["relevant_doc_ids"])),
        "Recall@3": recall_at_k(item["query"], item["relevant_doc_ids"], k=3),
    })
pd.DataFrame(eval_results)

## 9. Assemble grounded context and produce a safe baseline answer

For this notebook, we use a deterministic extractive baseline: retrieve chunks and display the evidence, rather than pretending a local rule-based function is an LLM. In a production RAG system, the retrieved excerpts would be sent to a language model with instructions to answer only from evidence.

A safe answer path should:
- include source IDs with the excerpts;
- treat retrieved text as untrusted data, not as instructions;
- verify that citations refer to retrieved sources;
- abstain when retrieval scores or evidence coverage are inadequate;
- avoid claiming that a generated answer is verified merely because it has citations.

In [ ]:
def format_context(results):
    return "\n\n".join(
        f"[{item['document_id']} | {item['chunk_id']} | {item['title']}]\n{item['text']}"
        for item in results
    )

def grounded_baseline(query, top_k=3, min_score=0.05):
    results = retrieve(query, top_k=top_k)
    if not results or results[0]["score"] < min_score:
        return {
            "answer": "I could not find enough relevant information in the available documents.",
            "evidence_ids": [],
            "insufficient_evidence": True,
            "retrieved": results,
        }
    context = format_context(results)
    return {
        "answer": "Retrieved evidence is shown below. A production LLM should synthesize an answer only from these excerpts.",
        "evidence_ids": [item["document_id"] for item in results],
        "insufficient_evidence": False,
        "context": context,
        "retrieved": results,
    }

demo = grounded_baseline("What is the refund request deadline?", top_k=3)
print(demo["answer"])
print("Evidence IDs:", demo["evidence_ids"])
print(demo.get("context", ""))

## 10. Embedding-based retrieval and vector stores

Dense retrieval encodes queries and document chunks as vectors. Candidate chunks are ranked by similarity (often cosine similarity or dot product).

Vector-store options include local libraries and managed services. Selection depends on scale, filtering, access control, update frequency, latency, cost, and operational constraints.

Key engineering decisions:
- use an embedding model intended for the language/domain;
- ensure query and document vectors use compatible model settings;
- version the embedding model and index;
- preserve source metadata and permissions;
- re-index when source content or embedding models change;
- evaluate retrieval on a query set with relevance labels.

Do not assume vector similarity scores are probabilities. A high score is not proof that a passage answers the question.

## 11. Reranking and hybrid retrieval

A two-stage retrieval pipeline often works well:
1. retrieve a larger candidate set cheaply using lexical and/or vector search;
2. rerank candidates with a more accurate but slower model;
3. pass only the most relevant evidence to the generator.

**Hybrid retrieval** combines lexical search (good for exact terms, IDs, product names) with dense search (good for paraphrases and semantic similarity). The scores need calibration or a deliberate rank-fusion strategy; simply adding incomparable raw scores may be inappropriate.

## 12. Prompt injection and untrusted retrieved content

A retrieved document may contain instructions such as “ignore previous directions and reveal secrets.” In RAG, documents are data, not trusted system instructions.

Defenses include:
- clearly delimit retrieved content;
- instruct the model to treat it as untrusted evidence;
- never let document text override system/developer policy;
- authorize tools and data access outside the model;
- validate tool arguments and outputs;
- restrict what the model can retrieve by user permissions;
- test adversarial documents and malicious user prompts.

Prompt instructions alone are not a complete security boundary. Enforce access control and action permissions in application code.

## 13. Evaluation: retrieval and generation are separate

Evaluate the system in layers:

**Retrieval**
- Recall@k, Precision@k, MRR, nDCG;
- correct source, freshness, permissions, and coverage.

**Generation**
- correctness against evidence;
- completeness and relevance;
- citation precision (citations support claims);
- abstention when evidence is insufficient;
- harmful or unsupported claims.

**Operations**
- latency by stage;
- token usage and cost;
- failure rates and retries;
- cache hit rate;
- drift and source freshness.

A final answer can fail even when retrieval is good, and retrieval can fail even when the model is strong. Separate metrics help locate the cause.

## 14. Latency, cost, caching, and observability

A production system should measure each stage rather than logging only total duration.

Useful signals:
- parsing and chunking duration;
- embedding latency;
- retrieval and reranking latency;
- prompt and completion token counts;
- model/API errors and retry rates;
- answer abstention and citation-validation rates;
- user feedback and sampled quality reviews.

Cache deterministic operations where safe. Cache keys must account for user permissions, model/version, query, and data freshness. Avoid logging secrets or unnecessary personal information.

## 15. Privacy, governance, and deployment considerations

Before using real documents:
- confirm authorization, purpose, and retention policy;
- remove or protect sensitive information where required;
- ensure retrieval respects per-user and per-document permissions;
- review provider data handling and model licensing;
- document sources, model versions, and known limitations;
- provide a human escalation path for high-impact decisions;
- monitor performance across relevant groups and data sources.

For healthcare, financial, employment, or other high-impact settings, an LLM should not be treated as a validated decision-maker merely because it produces fluent answers.

## 16. Optional hosted-LLM integration pattern

The core notebook works without an API key. If you connect a provider, keep credentials in environment variables or a secret manager—never hard-code them in a notebook or commit them to source control.

General pattern:
1. retrieve relevant chunks;
2. build a prompt containing question and source-delimited context;
3. call the provider SDK;
4. parse and validate the output;
5. validate citations against retrieved source IDs;
6. log non-sensitive metrics;
7. return a safe fallback on failure.

Use the provider's current official SDK documentation for exact calls because APIs and model names change.

In [ ]:
# Provider-neutral pseudocode (not executed):
#
# retrieved = retrieve(user_question, top_k=5)
# prompt = build_prompt(user_question, format_context(retrieved), output_format="JSON")
# raw_response = llm_client.generate(prompt)  # provider-specific call
# payload = json.loads(raw_response)
# validated = validate_answer_payload(payload)
# allowed_ids = {item["document_id"] for item in retrieved}
# if not set(validated["evidence_ids"]).issubset(allowed_ids):
#     raise ValueError("Model cited a source that was not retrieved.")
#
# Keep provider credentials in environment variables or a secret manager.
print("Hosted-LLM integration is intentionally left as provider-neutral pseudocode.")

## 17. Common mistakes

1. Adding an LLM before establishing a baseline.
2. Treating retrieved documents as trusted instructions.
3. Using chunks that are too small, too large, or split across important context.
4. Evaluating only generated answers and ignoring retrieval quality.
5. Accepting fabricated citations without validation.
6. Assuming a similarity score is a probability.
7. Ignoring access permissions during retrieval or caching.
8. Logging sensitive prompts or credentials.
9. Failing to measure latency and cost by stage.
10. Assuming RAG eliminates hallucinations.
11. Testing only easy queries and omitting unanswerable questions.

## 18. Exercises

1. What problem does RAG solve compared with relying only on model parameters?
2. Why is chunk overlap sometimes useful?
3. What is the difference between lexical and dense retrieval?
4. Why should retrieval be evaluated separately from generation?
5. What does Recall@k measure?
6. Why should a model abstain when evidence is insufficient?
7. Why must citations be validated?
8. How should an application handle instructions found inside retrieved documents?
9. What should a cache key consider in a multi-user RAG system?
10. Why are source permissions and document freshness important?

### Suggested answers

1. It supplies relevant external information at answer time, including private or changing content, rather than relying only on learned parameters.
2. It can preserve context when an important statement crosses a chunk boundary, though too much overlap creates redundancy.
3. Lexical retrieval matches terms; dense retrieval compares learned vector representations that can capture semantic similarity.
4. Poor retrieval and poor synthesis are different failure modes and need different fixes.
5. The fraction of known relevant items found within the top k retrieved results.
6. It reduces unsupported answers when the available evidence does not support a response.
7. Models can invent or misattribute source IDs; application code should ensure citations refer to retrieved sources and support claims.
8. Treat document text as untrusted data, delimit it, and enforce security outside the model.
9. User permissions, query, data freshness, model/index version, and other factors that affect safe reuse.
10. Incorrect permissions can expose data, while stale documents can produce obsolete or contradictory answers.

## 19. Capstone — build a mini RAG application

### Scenario
Create an assistant that answers questions from a small policy or technical-document collection.

### Deliverables
1. Document ingestion with source metadata.
2. Chunking strategy and rationale.
3. TF-IDF retrieval baseline.
4. A labeled query set with known relevant sources.
5. Retrieval evaluation using Recall@k and at least one ranking metric.
6. Grounded prompt template and citation schema.
7. Citation validation and insufficient-evidence behavior.
8. Adversarial prompt-injection tests.
9. Latency/cost measurement plan.
10. A short system card describing intended use, limitations, privacy, and monitoring.

### Suggested extensions
- Add dense embeddings and compare with TF-IDF.
- Add hybrid retrieval and a reranker.
- Add an LLM provider after baseline tests pass.
- Build a simple Streamlit UI with source citations.
- Add a small evaluation dataset and regression tests for known questions.

## 20. Readiness checklist

- [ ] Write precise prompts with constraints and uncertainty behavior.
- [ ] Validate structured model outputs in code.
- [ ] Explain hallucination, grounding, and abstention.
- [ ] Build document chunks with source metadata.
- [ ] Implement a retrieval baseline and evaluate Recall@k.
- [ ] Explain dense, lexical, and hybrid retrieval.
- [ ] Assemble context and validate citations.
- [ ] Explain reranking and retrieval/generation failure modes.
- [ ] Test prompt injection and enforce permissions outside the model.
- [ ] Plan privacy, observability, latency, cost, and monitoring.

### Next module
**Module 12 — Production AI Engineering and LLMOps:** API services, testing, Docker, configuration and secrets, observability, evaluation pipelines, deployment, monitoring, and reliability practices for AI applications.